In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
# Define columns needed and create dataframes from IPEDS 2011 and 2021
cols = ["UNITID", "CIPCODE", "AWLEVEL", "CTOTALT", "CTOTALM", "CTOTALW"]
c2011_clean = pd.read_csv(
    r"..\data\raw\c2011_a.csv", 
    usecols=cols,
    dtype={"CIPCODE":str}
)

In [3]:
c2024_clean = pd.read_csv(
    r"..\data\raw\c2024_a.csv",
    #r"C:\capstone_data\raw\c2024_a.csv", 
    usecols=cols,
    dtype={"CIPCODE":str}
)

In [4]:
# Dictionary for renaming columns
rename_dict = {
    "UNITID" : "unitid"
    ,"CIPCODE" : "cipcode"
    ,"AWLEVEL" : "credlev"
    ,"CTOTALT" : "completions"
    ,"CTOTALM" : "completions_men"
    ,"CTOTALW" : "completions_women"
}

In [5]:
# Rename the columns
c2011_clean = c2011_clean.rename(columns=rename_dict)
c2024_clean = c2024_clean.rename(columns=rename_dict)

In [6]:
print(c2011_clean.columns == c2024_clean.columns)

[ True  True  True  True  True  True]


In [7]:
# Set years
c2011_clean["year"] = 2011
c2024_clean["year"] = 2024

In [8]:
# "Stack" the two sets of data so that 2011 sits on top of 2021
completions = pd.concat([c2011_clean, c2024_clean], ignore_index=True)
completions.shape

(573515, 7)

In [9]:
c2011_clean["credlev"].value_counts().sort_index()
c2024_clean["credlev"].value_counts().sort_index()

credlev
2      29688
3      48741
4       2445
5     104090
6      14936
7      45516
8       5378
17     13469
18      3171
19       372
20      4543
21     35358
Name: count, dtype: int64

In [10]:
# Create a dictionary for the credlev codes, limiting it to the ones we are interested in
level_map = {
    2 : "Certificate"
    ,3 : "Certificate"
    ,5 : "Certificate"
    ,4 : "Associate"
    ,6 : "Bachelor"
}
completions["cred_desc"] = completions["credlev"].map(level_map)

# Format ipeds cip code to xxxx dtype str format. Maintain cip code 99 for accuracy.
def ipeds_to_4digit(cip):
    s = str(cip)
    digits = "".join(re.findall(r"\d+", s))    # Extract the digits
    if digits == "99":
        return "9900"                         # Special: cip 99 to 9900
    digits = digits.zfill(6)
    return digits [:4]

completions["cip4"] = completions["cipcode"].apply(ipeds_to_4digit)

In [11]:
# Checkpoint to see if everything is looking good.
completions["year"].value_counts()

year
2024    307707
2011    265808
Name: count, dtype: int64

In [12]:
completions["credlev"].value_counts()

credlev
5     207391
3      97232
7      81361
2      58807
21     35358
1      25020
17     24086
6      20051
8       8511
4       5388
18      5099
20      4543
19       668
Name: count, dtype: int64

In [13]:
completions.head(8)

,unitid,cipcode,credlev,completions,completions_men,completions_women,year,cred_desc
0,100636,09.0999,3,66,34,32,2011,Certificate
1,100636,10.0105,3,547,398,149,2011,Certificate
2,100636,11.0101,3,69,65,4,2011,Certificate
3,100636,11.0401,3,915,705,210,2011,Certificate
4,100636,13.0499,3,269,144,125,2011,Certificate
5,100636,13.9999,3,1486,1244,242,2011,Certificate
6,100636,13.9999,4,993,812,181,2011,Associate
7,100636,15.0303,3,1012,913,99,2011,Certificate


In [14]:
completions.tail(8)

,unitid,cipcode,credlev,completions,completions_men,completions_women,year,cred_desc
573507,500537,99,2,0,0,0,2024,Certificate
573508,500537,99,21,0,0,0,2024,NaN
573509,500555,12.0401,2,26,4,22,2024,Certificate
573510,500555,12.0402,2,15,12,3,2024,Certificate
573511,500555,12.0409,2,15,1,14,2024,Certificate
573512,500555,12.0410,2,0,0,0,2024,Certificate
573513,500555,12.0413,2,4,1,3,2024,Certificate
573514,500555,99,2,60,18,42,2024,Certificate


In [15]:
completions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573515 entries, 0 to 573514
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   unitid             573515 non-null  int64 
 1   cipcode            573515 non-null  object
 2   credlev            573515 non-null  int64 
 3   completions        573515 non-null  int64 
 4   completions_men    573515 non-null  int64 
 5   completions_women  573515 non-null  int64 
 6   year               573515 non-null  int64 
 7   cred_desc          388869 non-null  object
dtypes: int64(6), object(2)
memory usage: 35.0+ MB


In [16]:
completions.dtypes

unitid                int64
cipcode              object
credlev               int64
completions           int64
completions_men       int64
completions_women     int64
year                  int64
cred_desc            object
dtype: object

completions["cip4"].unique()

In [17]:
completions[["credlev", "cred_desc"]].value_counts()

credlev  cred_desc  
5        Certificate    207391
3        Certificate     97232
2        Certificate     58807
6        Bachelor        20051
4        Associate        5388
Name: count, dtype: int64

In [18]:
# Map completions credlev to match scorecard_tn
map_ipeds_to_fos = {
    2 : 1     # certificate < 1 year
    ,3 : 1    # certificate 1 - 2 years
    ,5 : 1    # certificate 2 - 4 years
    ,4 : 2    # associate
    ,6 : 3    # bachelor
}
completions["credlev"] = completions["credlev"].map(map_ipeds_to_fos)

In [19]:
# Map completions cred_desc to match scorecard_tn
level_map = {
    1 : "Certificate"
    ,2 : "Assoicate"
    ,3 : "Bachelor"
}
completions["cred_desc"] = completions["credlev"].map(level_map)

In [20]:
completions[["credlev", "cred_desc"]].value_counts()

credlev  cred_desc  
1.0      Certificate    363430
3.0      Bachelor        20051
2.0      Assoicate        5388
Name: count, dtype: int64

In [21]:
completions.dtypes

unitid                 int64
cipcode               object
credlev              float64
completions            int64
completions_men        int64
completions_women      int64
year                   int64
cred_desc             object
dtype: object

In [22]:
completions[["credlev", "cred_desc"]].value_counts().sort_index()

credlev  cred_desc  
1.0      Certificate    363430
2.0      Assoicate        5388
3.0      Bachelor        20051
Name: count, dtype: int64

# Convert dtypes to prepare for PBI
completions["unitid"] = completions["unitid"].astype(str)
completions["cip4"] = completions["cip4"].astype(str)
completions["credlev"] = completions["credlev"].astype(str)
# completions["year"] = completions["year"].astype(int64)

In [23]:
completions.dtypes

unitid                 int64
cipcode               object
credlev              float64
completions            int64
completions_men        int64
completions_women      int64
year                   int64
cred_desc             object
dtype: object

In [24]:
completions.isna().mean().sort_values(ascending=False)

credlev              0.321955
cred_desc            0.321955
cipcode              0.000000
unitid               0.000000
completions          0.000000
completions_men      0.000000
completions_women    0.000000
year                 0.000000
dtype: float64

In [25]:
completions['unitid'].nunique()

8477

completions['cip4'].nunique()

In [26]:
# Drop columns before exporting to PBI.

#completions_export = completions.drop(columns=["cipcode", "cred_desc"])
completions_export = completions.drop(columns=["cred_desc"])
completions_export.head(5)

,unitid,cipcode,credlev,completions,completions_men,completions_women,year
0,100636,09.0999,1.0,66,34,32,2011
1,100636,10.0105,1.0,547,398,149,2011
2,100636,11.0101,1.0,69,65,4,2011
3,100636,11.0401,1.0,915,705,210,2011
4,100636,13.0499,1.0,269,144,125,2011


In [27]:
### Keep cip6 as string
def normalize_cip6(series):
    return series.astype(str).str.strip()

In [28]:
##### 
def normalize_unitid(series):
    return series.astype(str).str.replace('.0', '', regex=False).str.strip()

In [29]:
##### Create cip4 as an aggregate of cip6 last 2 digits
def cip6_to_cip4(series):
    return (
        series.astype(str)
        .str.replace('.', '', regex=False)
        .str.strip()
        .str[:4]
        .str.zfill(4)
    )

In [30]:
##### Normalize columns
completions_export['unitid_n'] = normalize_unitid(completions_export['unitid'])
completions_export['cip6'] = normalize_cip6(completions_export['cipcode'])
completions_export['cip4'] = cip6_to_cip4(completions_export['cip6'])

In [31]:
#####
completions_export['credlev_n'] = pd.to_numeric(
    completions_export['credlev'], errors='coerce'
).astype('Int64')

In [32]:
completions_export = completions_export.dropna(subset=['credlev_n'])

In [33]:
##### Aggregate to cip4 - add up all completions in a cip6 to the cip4 level
completions_cip4_agg = (
    completions_export
    .groupby(
        ['unitid_n', 'cip4', 'credlev_n', 'year'],
        as_index=False
    )
    .agg({
        'completions' : 'sum',
        'completions_men' : 'sum',
        'completions_women' : 'sum'
    })
)

In [34]:
# completions_cip4_agg = completions_cip4_agg[completions_cip4_agg['completions'] > 0]

In [36]:
completions_cip4_agg = completions_cip4_agg[
    (completions_cip4_agg['completions'] > 0) |
    (completions_cip4_agg['completions_men'] > 0) |
    (completions_cip4_agg['completions_women'] > 0)
     ]

In [37]:
##### Build composite key for PBI join
completions_cip4_agg['credlev_str'] = completions_cip4_agg['credlev_n'].astype(str)

completions_cip4_agg['composite_key'] = (
    completions_cip4_agg['unitid_n'] + "|" +
    completions_cip4_agg['cip4'] + "|" +
    completions_cip4_agg['credlev_str']
)

In [38]:
#####
completions_cip4_agg.to_csv(
    "completions_cip4_agg.csv",
    index=False,
    encoding="utf-8"
)

In [39]:
#####
completions_cip4_agg.head()

,unitid_n,cip4,credlev_n,year,completions,completions_men,completions_women,credlev_str,composite_key
0,100636,0099,1,2011,18270,13847,4423,1,100636|0099|1
1,100636,0099,2,2011,993,812,181,2,100636|0099|2
2,100636,0909,1,2011,66,34,32,1,100636|0909|1
3,100636,1001,1,2011,547,398,149,1,100636|1001|1
4,100636,1101,1,2011,69,65,4,1,100636|1101|1


completions_export['credlev'].unique()

completions_export.dtypes

# Create a composite key because Power BI won't allow the desired three way join.
completions_export['composite_key'] = (
    completions_export['unitid'].astype(str) + "|" +
    completions_export['cip4'].astype(str) + "|" +
    completions_export['credlev'].astype(str)
)

# Get rid of the old composite_key
if 'composite_key' in completions_export.columns:
    completions_export = completions_export.drop(columns=['composite_key'])

# Build a new composite_key using normalized columns
completions_export['unitid_n'] = (
   completions_export['unitid'].astype(str)
    .str.replace('.0', "", regex=False)
    .str.strip()
)

def canonical_cip4(series):
    return (
        series.astype(str)
              .str.replace('.', '', regex=False)
              .str.replace(' ', '', regex=False)
              .str.strip()
              .str[:4]
              .str.zfill(4)
    )

completions_export['cip4_n'] = canonical_cip4(completions_export['cip4'])

completions_export['credlev'] = completions_export['credlev'].fillna(0)

In [ ]:
# completions_export['credlev'] = completions_export['credlev'].astype(int)

completions_export['composite_key'] = (
    completions_export['unitid_n'] + "|" +
    completions_export['cip4_n'] + "|" +
    completions_export['credlev'].astype(str)
)

completions_export.head(5)

completions_export.to_csv(
    "completions_export.csv",
    index=False,
    encoding="utf-8"
)